# Benchmark Eval (No Merge)

다음 3개 모델을 동일 파이프라인으로 평가합니다.

- `alwaysgood/qwen3-st2`
- `alwaysgood/qwen35-st2`
- `alwaysgood/gemma4-st2`

평가 항목:
- Validation PPL
- EN 벤치: `mmlu`, `hellaswag`, `arc_easy`, `arc_challenge`, `winogrande`
- KO 벤치: `kmmlu`, `kobest_boolq`, `kobest_copa`, `kobest_hellaswag`


## 0. 환경 설정

커널 Python 환경에 직접 설치합니다. (mergekit 설치/머지 로직 없음)

In [ ]:
import sys

# core
!{sys.executable} -m pip install -q unsloth unsloth-zoo datasets trl hydra-core omegaconf sentencepiece numpy pandas tabulate
# eval
!{sys.executable} -m pip install -q lm-eval

# (선택) private repo 접근이 필요하면 로그인
# from huggingface_hub import notebook_login
# notebook_login()


In [ ]:
import unsloth  # transformers 보다 먼저 import
import os
import gc
import json
import math
import glob
import re
from pathlib import Path
from collections import Counter

import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset, load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
from lm_eval import simple_evaluate
from lm_eval.models.huggingface import HFLM

MODELS = [
    "alwaysgood/qwen3-st2",
    "alwaysgood/qwen35-st2",
    "alwaysgood/gemma4-st2",
]

# tokenization/chunking 기준 tokenizer (val set 준비용)
TOKENIZER_SOURCE = MODELS[0]

EN_TASKS  = ["mmlu", "hellaswag", "arc_easy", "arc_challenge", "winogrande"]
KO_TASKS  = ["kmmlu", "kobest_boolq", "kobest_copa", "kobest_hellaswag"]
ALL_TASKS = EN_TASKS + KO_TASKS

MAX_SEQ_LENGTH = 2048
SEED = 42
VAL_RATIO = 0.02
LIMIT = 400
RUN_PPL = False   # True 로 바꾸면 PPL 계산 포함
PPL_BATCH_SIZE = 32  # RUN_PPL=True 일 때만 사용 (OOM 시 8/4/2/1)

EVAL_ROOT = Path("./eval_results_bench")
EVAL_ROOT.mkdir(exist_ok=True)

print("models:")
for m in MODELS:
    print(" -", m)


## 1. Val set 준비 (기존 benchmark 계산 로직 유지)

In [ ]:
DATASET_REPOS = [
    "alwaysgood/korean-financial-cpt",
    "alwaysgood/korean_rlhf_content_filtered",
]

TEXT_FIELDS = ["content", "contents", "text", "document", "body"]
MIN_CHARS = 20

def normalize_text(text):
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()

def extract_text(row):
    text = None
    for f in TEXT_FIELDS:
        if f in row and row[f] is not None:
            text = row[f]
            break
    if text is None:
        return None
    if isinstance(text, list):
        text = "\n".join(str(x) for x in text)
    elif isinstance(text, dict):
        text = json.dumps(text, ensure_ascii=False)
    elif not isinstance(text, str):
        text = str(text)
    text = normalize_text(text)
    return text if text else None

def split_paragraphs(t):
    paras = [p.strip() for p in re.split(r"\n\s*\n+", t) if p.strip()]
    return paras or [t]

def chunk_boundary_first(text, tok, max_body, sep_ids):
    paras = split_paragraphs(text)
    chunks, cur = [], []
    for p in paras:
        pids = tok.encode(p, add_special_tokens=False)
        if not pids:
            continue
        if len(pids) > max_body:
            if cur:
                chunks.append(cur)
                cur = []
            for s in range(0, len(pids), max_body):
                chunks.append(pids[s:s + max_body])
            continue
        extra = (len(sep_ids) if cur else 0) + len(pids)
        if len(cur) + extra <= max_body:
            if cur:
                cur.extend(sep_ids)
            cur.extend(pids)
        else:
            chunks.append(cur)
            cur = list(pids)
    if cur:
        chunks.append(cur)
    return chunks

tok = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE, trust_remote_code=True)
eos_id = tok.eos_token_id
para_sep_ids = tok.encode("\n\n", add_special_tokens=False)
max_body = MAX_SEQ_LENGTH - 1

records, skipped = [], 0
for repo in DATASET_REPOS:
    snap = snapshot_download(repo, repo_type="dataset")
    jsonl_files = sorted(glob.glob(os.path.join(snap, "**/*.jsonl"), recursive=True))
    print(f"[{repo}] {len(jsonl_files)} jsonl files")

    for f in tqdm(jsonl_files, desc=repo):
        ds = load_dataset("json", data_files=f, split="train")
        for row in ds:
            text = extract_text(row)
            if text is None or len(text) < MIN_CHARS:
                skipped += 1
                continue
            chunks = chunk_boundary_first(text, tok, max_body, para_sep_ids)
            total_chunks = len(chunks)
            for i, ids in enumerate(chunks):
                if not ids:
                    continue
                is_end = i == total_chunks - 1
                if is_end and ids[-1] != eos_id:
                    ids = ids + [eos_id]
                if len(ids) > MAX_SEQ_LENGTH:
                    ids = ids[:MAX_SEQ_LENGTH]
                records.append({"input_ids": ids, "labels": list(ids), "source": repo})

full = Dataset.from_list(records)
val_size = max(1, int(len(full) * VAL_RATIO))
split = full.train_test_split(test_size=val_size, seed=SEED, shuffle=True)
val_ds = split["test"]

print(f"\nrecords total : {len(full):,} (skipped {skipped})")
print(f"val_ds        : {len(val_ds):,} rows")
src_counts = Counter(val_ds["source"])
for s, c in src_counts.items():
    print(f"  - {s}: {c:,}")


## 2. 모델별 Benchmark 계산

In [ ]:
def evaluate_model(name, model_path, val_ds):
    print(f"\n{'='*60}\n▶ EVAL  {name}  ({model_path})\n{'='*60}")

    for _v in ("model", "tok_local", "hflm", "bench", "_hf_tok"):
        if _v in globals():
            del globals()[_v]
    gc.collect()
    torch.cuda.empty_cache()

    model, tok_local = FastLanguageModel.from_pretrained(
        model_name=str(model_path),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
        dtype=torch.bfloat16,
    )
    FastLanguageModel.for_inference(model)

    # ----- PPL (optional) -----
    if RUN_PPL:
        pad_id = tok_local.pad_token_id
        if pad_id is None:
            pad_id = tok_local.eos_token_id

        n = len(val_ds)
        order = sorted(range(n), key=lambda i: len(val_ds[i]["input_ids"]))
        total_loss, total_tokens = 0.0, 0

        with torch.no_grad():
            for bstart in tqdm(range(0, n, PPL_BATCH_SIZE), desc=f"PPL[{name}]"):
                batch_idx = order[bstart:bstart + PPL_BATCH_SIZE]
                seqs = [val_ds[i]["input_ids"] for i in batch_idx]
                labs = [val_ds[i]["labels"] for i in batch_idx]
                maxlen = max(len(s) for s in seqs)

                input_ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
                labels = torch.full((len(seqs), maxlen), -100, dtype=torch.long)
                attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)

                for j, (s, l) in enumerate(zip(seqs, labs)):
                    input_ids[j, :len(s)] = torch.tensor(s, dtype=torch.long)
                    labels[j, :len(l)] = torch.tensor(l, dtype=torch.long)
                    attn[j, :len(s)] = 1

                input_ids = input_ids.to(model.device)
                labels = labels.to(model.device)
                attn = attn.to(model.device)

                out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
                ntok = (labels != -100).sum().item()
                valid = max(ntok - len(seqs), 1)
                total_loss += out.loss.item() * valid
                total_tokens += valid

                del input_ids, labels, attn, out

        avg = total_loss / max(total_tokens, 1)
        ppl_result = {"ppl": math.exp(avg), "loss": avg, "tokens": total_tokens, "rows": n}
        (EVAL_ROOT / f"{name}_ppl.json").write_text(json.dumps(ppl_result, indent=2, ensure_ascii=False))
        print(f"  PPL = {ppl_result['ppl']:.4f}")
    else:
        print("  PPL skipped (RUN_PPL=False)")

    # ----- LM-Eval -----
    import transformers as _tf
    _hf_tok = tok_local
    while not isinstance(_hf_tok, (_tf.PreTrainedTokenizer, _tf.PreTrainedTokenizerFast)):
        inner = getattr(_hf_tok, "tokenizer", None)
        if inner is None or inner is _hf_tok:
            _hf_tok = _tf.AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
            break
        _hf_tok = inner

    hflm = HFLM(pretrained=model, tokenizer=_hf_tok, batch_size="auto")
    bench = simple_evaluate(
        model=hflm,
        tasks=ALL_TASKS,
        limit=LIMIT,
    )

    out_dir = EVAL_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "results_unsloth.json").write_text(json.dumps(bench, indent=2, default=str, ensure_ascii=False))
    print(f"  saved: {out_dir / 'results_unsloth.json'}")

    del model, tok_local, hflm, bench
    gc.collect()
    torch.cuda.empty_cache()

for model_id in MODELS:
    name = model_id.split("/")[-1]
    evaluate_model(name, model_id, val_ds)

print("\n✅ 전체 모델 평가 완료")


## 3. 결과 집계 표

In [ ]:
def latest_results_json(folder: Path):
    if not folder.exists():
        return None
    cands = sorted(folder.rglob("results_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    return json.loads(cands[0].read_text(encoding="utf-8")) if cands else None

def task_score(results_json, task):
    if not results_json or "results" not in results_json:
        return None
    r = results_json["results"]
    if task in r:
        for k, v in r[task].items():
            if isinstance(v, (int, float)) and "acc" in k and "stderr" not in k:
                return v

    sub = [v for k, v in r.items() if k.startswith(task) and k != task]
    accs = []
    for s in sub:
        for kk, vv in s.items():
            if isinstance(vv, (int, float)) and "acc" in kk and "stderr" not in kk:
                accs.append(vv)
                break
    return sum(accs) / len(accs) if accs else None

def ppl_for(name):
    p = EVAL_ROOT / f"{name}_ppl.json"
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8")).get("ppl")

rows = []
for model_id in MODELS:
    name = model_id.split("/")[-1]
    res = latest_results_json(EVAL_ROOT / name)
    rows.append({
        "model": model_id,
        "ppl": ppl_for(name),
        **{t: task_score(res, t) for t in ALL_TASKS},
    })

df = pd.DataFrame(rows)
df["EN_avg"] = df[EN_TASKS].mean(axis=1)
df["KO_avg"] = df[KO_TASKS].mean(axis=1)

cols = ["model", "ppl", "EN_avg", "KO_avg"] + EN_TASKS + KO_TASKS
df = df[cols]

print(df.to_markdown(index=False, floatfmt=".4f"))
df.to_csv(EVAL_ROOT / "bench_comparison.csv", index=False)
print(f"\n💾 saved: {EVAL_ROOT / 'bench_comparison.csv'}")
